# LCEL (LangChain Expression Language)

# 選擇模型
請自由任意選擇下面兩個模型之一來進行範例  

## Gemini

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

API_KEY = "這邊請改成你自己的API_KEY值"
model_name = 'gemini-2.5-flash'

llm = ChatGoogleGenerativeAI(
    model=model_name,
    google_api_key=API_KEY,
    temperature=0
)

## LM Studio 或 Ollama

In [ ]:
from langchain_openai import ChatOpenAI
model_name = 'gemma-3-4b-it'  # 指定模型名稱，模型名稱會根據下載的模型不同而改變

# 根據使用LM Studio或Ollama來選擇適當的伺服器URL
base_url = 'http://localhost:1234/v1'  # LM Studio 本地伺服器的URL
# base_url = 'http://localhost:11434/v1' # Ollama 本地伺服器的URL

llm = ChatOpenAI(
    model=model_name,
    openai_api_key="not-needed",
    openai_api_base=base_url,
    temperature=0
)

# Chain

## 一個簡單的Chain

In [10]:
from langchain_core.messages import HumanMessage
messages = [
    HumanMessage("簡短的說明機器學習的定義")
]

response = llm.invoke(messages)
print(response.content)

機器學習 (Machine Learning) 是一種讓電腦**無需明確編程，就能從數據中學習並做出預測或決策**的方法。

簡單來說，就是給電腦大量數據，讓它自己找出規律和模式，然後用這些規律來解決問題。

例如：

*   **垃圾郵件過濾:** 機器學習模型會學習識別垃圾郵件的特徵（例如關鍵詞、發送者等），並自動將它們過濾掉。
*   **推薦系統:**  機器學習模型會根據你的瀏覽歷史和購買記錄，推薦你可能感興趣的商品或服務。

希望這個簡短說明對您有幫助！



In [16]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()
parser.invoke(response)

'機器學習 (Machine Learning) 是一種讓電腦**無需明確編程，就能從數據中學習並做出預測或決策**的方法。\n\n簡單來說，就是給電腦大量數據，讓它自己找出規律和模式，然後用這些規律來解決問題。\n\n例如：\n\n*   **垃圾郵件過濾:** 機器學習模型會學習識別垃圾郵件的特徵（例如關鍵詞、發送者等），並自動將它們過濾掉。\n*   **推薦系統:**  機器學習模型會根據你的瀏覽歷史和購買記錄，推薦你可能感興趣的商品或服務。\n\n希望這個簡短說明對您有幫助！\n'

In [15]:
chain = llm | parser
chain.invoke(messages)

'機器學習 (Machine Learning) 是一種讓電腦**無需明確編程，就能從數據中學習並做出預測或決策**的方法。\n\n簡單來說，就是給電腦大量數據，讓它自己找出規律和模式，然後用這些規律來解決問題。\n\n例如：\n\n*   **垃圾郵件過濾:** 機器學習模型會學習哪些郵件是垃圾郵件，並自動將新的郵件分類。\n*   **推薦系統:**  機器學習模型會根據你的瀏覽歷史和購買記錄，推薦你可能感興趣的商品或服務。\n\n希望這個簡短說明對您有幫助！\n'

## 加入 ChatPromptTemplate

In [23]:
from langchain_core.messages import HumanMessage, SystemMessage


def get_message(target_language, text):
    messages = [
        SystemMessage(f"將使用者的輸入翻譯成{target_language}，不需要解釋"),
        HumanMessage(text)
    ]
    return messages

chain.invoke(get_message("英文", "今天是下雨天!"))

"It's raining today!\n"

In [24]:
chain.invoke(get_message("日語", "今天是下雨天!"))

'今日は雨です！\n'

In [27]:
from langchain_core.prompts import ChatPromptTemplate
prompt_template = ChatPromptTemplate.from_messages([
        ("system", "將使用者的輸入翻譯成{target_language}，不需要解釋"),
        ("human", "{text}")
    ])

prompt_template.invoke({
    "target_language": "英文",
    "text": "今天是下雨天!"
})

ChatPromptValue(messages=[SystemMessage(content='將使用者的輸入翻譯成英文，不需要解釋', additional_kwargs={}, response_metadata={}), HumanMessage(content='今天是下雨天!', additional_kwargs={}, response_metadata={})])

In [28]:
chain_2 = prompt_template | llm | parser
chain_2.invoke({
        "target_language": "英文",
        "text": "今天是下雨天!"
    })


"It's raining today!\n"

## 使用串流 stream

In [30]:
for chunk in chain_2.stream({"target_language": "英文","text": "今天是下雨天!"}):
    print(chunk, end="")

It's raining today!


## 使用批次 batch

In [31]:
data = [
    {"target_language": "英文","text": "今天是下雨天!"},
    {"target_language": "日文","text": "今天是下雨天!"},
    {"target_language": "法文","text": "今天是下雨天!"}
]

chain_2.batch(data)

["It's raining today!\n", '今日は雨です！\n', "Il pleut aujourd'hui !\n"]

# 建立RAG Chain
範例會直接使用4.06a中建立的向量資料庫

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.output_parsers import StrOutputParser

system_prompt = '''
你是一位專業的電商客服代表。以下內容為公司內部的作業規範與應答準則：
{QA}

請根據上述規範，準確且禮貌地回答使用者的問題。  
請注意以下原則：
1. 回覆時不得提及或暗示任何內部文件、規章或系統資訊的存在。  
2. 僅能回應與公司平台、產品、訂單、客服流程等相關的問題。  
3. 若使用者的提問與平台或服務內容無關，請婉轉地表示無法協助，並引導回到與平台相關的主題。
'''

prompt_template = ChatPromptTemplate.from_messages(
        [
            ("system", system_prompt), 
            ("user", "{question}")    
        ]
    )

parser = StrOutputParser()
chain = prompt_template | llm | parser

In [58]:
from langchain_chroma import Chroma
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-large-zh-v1.5")

vector_store = Chroma(
            collection_name="clementshop_qa",
            embedding_function=embeddings,
            persist_directory="./chroma_db"
        )

def get_data(question):
    docs =  vector_store.similarity_search(question, k=5)
    return {"QA": docs, "question": question}

In [59]:
doc = get_data("購物會開發票嗎？")
response = chain.invoke(doc)
print(response)

您好！ClementShop 的所有訂單都會開立合法電子發票喔！我們的系統會即時更新相關資訊，確保您的資料與訂單狀態保持最新。 

若您在操作過程中遇到任何問題，建議您可先確認網路連線狀況，或重新整理頁面再試一次。 我們建議您於完成操作後截圖保存，以便日後查詢紀錄或問題申訴使用。


In [60]:
from langchain_core.runnables import RunnableLambda

rag_chain = RunnableLambda(get_data) | prompt_template | llm | parser

In [62]:
result = rag_chain.invoke("購物會開發票嗎？")
print(result)

您好！ClementShop 的所有訂單都會開立合法電子發票喔！我們的系統會即時更新相關資訊，確保您的資料與訂單狀態保持最新。 

若您在操作過程中遇到任何問題，建議您可先確認網路連線狀況，或重新整理頁面再試一次。 我們建議您於完成操作後截圖保存，以便日後查詢紀錄或問題申訴使用。


In [63]:
for chunk in rag_chain.stream("購物會開發票嗎？"):
    print(chunk, end="")

您好！ClementShop 的所有訂單都會開立合法電子發票喔！我們的系統會即時更新相關資訊，確保您的資料與訂單狀態保持最新。 

若您在操作過程中遇到任何問題，建議您可先確認網路連線狀況，或重新整理頁面再試一次。 我們建議您於完成操作後截圖保存，以便日後查詢紀錄或問題申訴使用。

In [67]:
questions = [
    "購物會開發票嗎？",
    "我想要退貨",
    "訂購後東西多久可以收到"
]

results = rag_chain.batch(questions)

for question, result in zip(questions, results):
    print(f"問題: {question}")
    print(f"回答: {result}")
    print("--------------")

問題: 購物會開發票嗎？
回答: 您好！ClementShop 的所有訂單都會開立合法電子發票喔！我們的系統會即時更新相關資訊，確保您的資料與訂單狀態保持最新。 建議您於完成操作後截圖保存，以便日後查詢紀錄或問題申訴使用。
--------------
問題: 我想要退貨
回答: 您好！很高興為您服務。

為了協助您進行退貨申請，請登入我們的會員中心，在您的訂單詳情頁面中點擊「申請退貨」按鈕喔！ 系統會即時更新相關資訊，確保您的資料與訂單狀態保持最新。

若您在操作過程中遇到任何問題，建議您可先確認網路連線狀況，或重新整理頁面再試一次。 

祝您購物愉快！
--------------
問題: 訂購後東西多久可以收到
回答: 您好，感謝您的詢問！

依據您的訂單地區和商品狀況，通常出貨後 1-3 個工作天內可送達。您可以透過我們的會員中心即時查詢訂單進度，以便清楚掌握每一步狀態。 我們也會主動以電子郵件或簡訊通知您最新進度。 ClementShop 的系統會即時更新相關資訊，確保您的資料與訂單狀態保持最新。

請注意，部分作業可能因例假日或物流延誤而稍有影響，敬請諒解。
--------------
